# 04 – Pré-processamento para VOSviewer

## Resumo
Prepara os dados do JMOe para importação no VOSviewer, realizando:

1. **Padronização de autores e palavras-chave** – limpeza de caracteres especiais e normalização de plurais.
2. **Processamento geográfico** – extração e padronização de países/estados/cidades.
3. **Geração do CSV VOSviewer** – arquivo no formato esperado pela ferramenta.
4. **Mapa mundial de publicações** – choropleth interativo com Plotly.
5. **Top 10 países** – gráfico de barras com os países mais produtivos.

**Entrada:** `files_csv/artigos_jmoe_final.csv`  
**Saídas:** `files_csv/vosviewer_ready.csv` | `figures/mapa_global_publicacoes.png`


## Instalação de Dependências

In [ ]:
!pip install nltk unidecode plotly kaleido pandas -q


## Imports

In [ ]:
import pandas as pd
import ast
import re
import unicodedata
import plotly
import plotly.express as px
from unidecode import unidecode
from collections import Counter, defaultdict


## 1. Carregamento dos Dados

In [ ]:
# Caminho relativo ao repositório
CAMINHO_ENTRADA = 'files_csv/artigos_jmoe_final.csv'
df = pd.read_csv(CAMINHO_ENTRADA)

print(f'Shape: {df.shape}')
df.head()


## 2. Funções de Limpeza

In [ ]:
def clean_list(x):
    """
    Converte uma coluna que contém listas em string (ex: "['UEMA', 'UFPA']")
    para uma string separada por '; ' legível pelo VOSviewer.
    """
    if pd.isna(x): return ''
    try:
        lst = ast.literal_eval(x)
        return '; '.join(str(i).strip() for i in lst)
    except:
        return str(x)

def clean_keywords(x):
    """
    Limpa a coluna de palavras-chave:
    - Remove 'Index Terms' (comum em artigos IEEE).
    - Remove quebras de linha e barras invertidas residuais.
    - Retorna palavras-chave separadas por '; '.
    """
    if pd.isna(x): return ''
    x = str(x)
    x = re.sub(r'Index Terms', '', x, flags=re.I)
    x = re.sub(r'[\n\t\\]+', ' ', x)
    x = re.sub(r'\s+', ' ', x)
    parts = [p.strip(' .') for p in x.split(';') if p.strip(' .')]
    return '; '.join(parts)


## 3. Normalização de Plurais nas Palavras-Chave

Unifica variações como 'antenna' / 'antennas' para evitar termos duplicados no VOSviewer.

In [ ]:
# Aplica limpezas básicas
df['authors']        = df['autores'].apply(clean_list)
df['authors']        = df['authors'].apply(lambda x: unidecode(x) if isinstance(x,str) else x)
df['affiliations']   = df['university'].apply(clean_list)
df['Author Keywords'] = df['palavras_chave'].apply(clean_keywords)

# Coleta todas as palavras-chave para análise de frequência de case
todas_kws = []
for row in df['Author Keywords'].dropna():
    for kw in row.split(';'):
        kw = kw.strip()
        if kw: todas_kws.append(kw)

# Determina a forma de capitalização mais frequente para cada keyword
kw_case_freq = defaultdict(Counter)
for kw in todas_kws:
    kw_case_freq[kw.lower()][kw] += 1
best_case_kw   = {k: v.most_common(1)[0][0] for k,v in kw_case_freq.items()}
lowercased_kws = set(best_case_kw.keys())

# Constrói mapeamento plural -> singular
plural_to_singular = {}
for kw in lowercased_kws:
    if kw.endswith('s') and not kw.endswith('ss') and not kw.endswith('is'):
        if kw[:-1] in lowercased_kws:
            plural_to_singular[kw] = kw[:-1]
        elif kw.endswith('ies') and kw[:-3]+'y' in lowercased_kws:
            plural_to_singular[kw] = kw[:-3]+'y'
        elif kw.endswith('es') and kw[:-2] in lowercased_kws:
            plural_to_singular[kw] = kw[:-2]

def normalize_plurals_and_case(text):
    """
    Aplica a normalização de plurais e capitalização às palavras-chave de um artigo.
    Exemplo: 'antennas' -> 'Antenna'.
    """
    if not isinstance(text, str): return text
    normalized = []
    for kw in text.split(';'):
        kw = kw.strip()
        kw_lower = kw.lower()
        # Substitui plural pelo singular, se mapeado
        kw_lower = plural_to_singular.get(kw_lower, kw_lower)
        # Aplica a capitalização mais frequente
        normalized.append(best_case_kw.get(kw_lower, kw))
    return '; '.join(normalized)

df['Author Keywords'] = df['Author Keywords'].apply(normalize_plurals_and_case)


## 4. Processamento Geográfico

In [ ]:
def parse_geo_list(x):
    """
    Converte campo geográfico (país, estado, cidade) para lista Python.
    Trata strings formatadas como listas e strings separadas por vírgula.
    """
    if pd.isna(x): return []
    if isinstance(x, list): return x
    if isinstance(x, str):
        try:
            res = ast.literal_eval(x)
            if isinstance(res, list): return res
        except:
            x = x.strip('[]')
            return [e.strip() for e in x.split(',') if e.strip()]
    return []

# Converte colunas geográficas para listas
df['pais']   = df['pais'].apply(parse_geo_list)
df['estado'] = df['estado'].apply(parse_geo_list)
df['cidade'] = df['cidade'].apply(parse_geo_list)


In [ ]:
# Mapeamento estado -> país (para preencher campos 'pais' vazios)
map_estado_pais = {
    'Minas Gerais': 'Brazil', 'Goias': 'Brazil', 'Ceará': 'Brazil',
    'Paraná': 'Brazil',       'Pará': 'Brazil',  'Paraíba': 'Brazil',
    'Rio Grande do Norte': 'Brazil', 'Rio Grande do Sul': 'Brazil',
    'Bahia': 'Brazil',        'Santa Catarina': 'Brazil',
    'Distrito Federal': 'Brazil',    'São Paulo': 'Brazil',
    'Pernambuco': 'Brazil',   'Telangana': 'India', 'Maharashtra': 'India',
    'Punjab': 'India',        'Nanjing': 'China',  'Johor': 'Malaysia',
    'Lagos': 'Nigeria',       'Quebec': 'Canada',  'Pisa': 'Italy',
    # Adicione outros mapeamentos conforme os dados
}

def substituir_por_dicionario(row):
    """Completa a lista de países usando o estado como fallback."""
    pais_lista   = row['pais'] if isinstance(row['pais'], list) else []
    estado_lista = row['estado'] if isinstance(row['estado'], list) else []
    if pais_lista: return pais_lista
    return [map_estado_pais.get(e, 'Unknown') for e in estado_lista]

df['pais']      = df.apply(substituir_por_dicionario, axis=1)
df['countries'] = df['pais'].apply(
    lambda x: '; '.join(str(i).strip() for i in x if i) if isinstance(x,list) else str(x)
)


## 5. Geração do CSV para VOSviewer

In [ ]:
# Seleciona e renomeia colunas para o formato esperado pelo VOSviewer
vos = df[['authors','titulo','ano','affiliations','countries',
          'Author Keywords','Citações']].copy()
vos.columns = ['Authors','Title','Year','Affiliations',
               'Country','Author Keywords','Cited by']

CAMINHO_SAIDA_VOS = 'files_csv/vosviewer_ready.csv'
vos.to_csv(CAMINHO_SAIDA_VOS, index=False)
print(f' Arquivo VOSviewer salvo em: {CAMINHO_SAIDA_VOS}')
vos.head()


## 6. Análise Geográfica – Mapa Mundial de Publicações

In [ ]:
# Expande cada artigo para uma linha por país (para contar publicações por país)
df_paises = df.explode('pais')
df_paises = df_paises[df_paises['pais'].notna() & (df_paises['pais'] != '')]
df_paises['pais'] = df_paises['pais'].astype(str).str.strip()

# Padronização final de nomes de países (variações encontradas nos dados)
mapa_paises = {
    'Brasil': 'Brazil',      'Brazi': 'Brazil',
    'INDIA':  'India',       'P. R. China': 'China',
    'ROC':    'Taiwan',      'UK': 'United Kingdom',
    'KSA':    'Saudi Arabia','Türkiye': 'Turkey',
    'Perú':   'Peru',        'Algérie': 'Algeria',
    'Viet Nam': 'Vietnam',   'Morocco ': 'Morocco',
}
df_paises['pais'] = df_paises['pais'].replace(mapa_paises)

# Remove duplicatas (evita contar o mesmo país 2x no mesmo artigo)
df_paises = df_paises.drop_duplicates(subset=['_id','pais'])

contagem = df_paises['pais'].value_counts().reset_index()
contagem.columns = ['pais','publicacoes']
contagem['publicacoes'] = pd.to_numeric(contagem['publicacoes'], errors='coerce')
print(contagem.sort_values('publicacoes', ascending=False).head(20))


In [ ]:
# Categoriza países em faixas de publicações para o choropleth
def faixa_pub(x):
    if x == 1:    return '1'
    elif x <= 3:  return '2-3'
    elif x <= 9:  return '4-9'
    elif x <= 49: return '10-49'
    else:         return '50+'

contagem['faixa'] = contagem['publicacoes'].apply(faixa_pub)

fig = px.choropleth(
    contagem,
    locations='pais',
    locationmode='country names',
    color='faixa',
    category_orders={'faixa': ['1','2-3','4-9','10-49','50+']},
    color_discrete_map={
        '1':     '#f7f7f7',
        '2-3':   '#d9d9d9',
        '4-9':   '#9ecae1',
        '10-49': '#3182bd',
        '50+':   '#08519c'
    },
    title='Global Distribution of Publications',
    hover_name='pais',
    hover_data={'publicacoes': True, 'faixa': False}
)
fig.update_layout(
    title={'text':'Global Distribution of Publications',
           'x':0.5, 'xanchor':'center', 'font':dict(size=20)},
    font=dict(family='Times New Roman', size=16),
    legend=dict(title=dict(text='Publications', font=dict(size=18)),
                font=dict(size=16), x=0.84, y=0.90,
                xanchor='left', yanchor='top',
                bgcolor='rgba(255,255,255,0.8)'),
)
fig.write_image('figures/mapa_global_publicacoes.png',
                width=1600, height=900, scale=4)
fig.show()


## 7. Top 10 Países – Gráfico de Barras

In [ ]:
top10 = contagem.nlargest(10, 'publicacoes')
fig_bar = px.bar(
    top10.sort_values('publicacoes'),
    x='publicacoes', y='pais', orientation='h',
    title='Top 10 Contributing Countries',
    labels={'publicacoes': 'Number of Publications', 'pais': 'Country'}
)
fig_bar.update_layout(
    font=dict(family='Times New Roman', size=14),
    yaxis=dict(categoryorder='total ascending')
)
fig_bar.write_image('figures/top10_paises.png', scale=4)
fig_bar.show()
